In [1]:
import pandas as pd
import numpy as np
from transformers import pipeline

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../Outputs/merged_text_905.csv")
print(f"Loaded: {df.shape[0]} rows")
print(f"Posts with usable text: {df['has_text'].sum()}")

Loaded: 905 rows
Posts with usable text: 797


In [3]:
# Binary classifier: offensive vs not-offensive
# Trained on Twitter — domain match for social media
tox_model = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-offensive",
    top_k=None,
    truncation=True
)

# Quick test
print(tox_model("I'm so excited for the Super Bowl!"))
print(tox_model("you're such an idiot, shut up"))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4574.64it/s]


[[{'label': 'non-offensive', 'score': 0.8750926852226257}, {'label': 'offensive', 'score': 0.12490732222795486}]]
[[{'label': 'offensive', 'score': 0.9381088614463806}, {'label': 'non-offensive', 'score': 0.06189114227890968}]]


In [5]:
texts = df.loc[df["has_text"], "text_for_analysis"].tolist()

# Cardiff model has stricter token limits. Cut more aggressively.
# ~3 chars per token roughly, so 1200 chars = ~400 tokens, safely under 512
texts = [t[:1200] for t in texts]

print(f"Running Cardiff offensive model on {len(texts)} posts...")
results = tox_model(texts, batch_size=8)  # smaller batch size too
print(f"Got {len(results)} results")

print(f"\nSample result keys: {[item['label'] for item in results[0]]}")

Running Cardiff offensive model on 797 posts...
Got 797 results

Sample result keys: ['non-offensive', 'offensive']


In [6]:
# Drop existing columns if they exist (safe re-run)
cols_to_drop = ["cardiff_offensive", "cardiff_nonoffensive", "cardiff_is_toxic", "cardiff_label"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Extract offensive and non-offensive scores (handle both label naming conventions)
offensive_scores = []
nonoffensive_scores = []
for res in results:
    score_dict = {item["label"]: item["score"] for item in res}
    # Try common label names
    off = score_dict.get("offensive", score_dict.get("LABEL_1", 0.0))
    non = score_dict.get("non-offensive", score_dict.get("LABEL_0", 0.0))
    offensive_scores.append(off)
    nonoffensive_scores.append(non)

cardiff_df = pd.DataFrame({
    "cardiff_offensive": offensive_scores,
    "cardiff_nonoffensive": nonoffensive_scores
})
cardiff_df.index = df.index[df["has_text"]]
df = df.join(cardiff_df)

# Apply 0.5 threshold
THRESHOLD = 0.5
df["cardiff_is_toxic"] = None
df["cardiff_label"] = None

mask = df["has_text"]
df.loc[mask, "cardiff_is_toxic"] = df.loc[mask, "cardiff_offensive"] > THRESHOLD
df.loc[mask & (df["cardiff_offensive"] > THRESHOLD), "cardiff_label"] = "offensive"
df.loc[mask & (df["cardiff_offensive"] <= THRESHOLD), "cardiff_label"] = "not_offensive"

print(df[["student_id", "text_source", "cardiff_label", "cardiff_offensive"]].head())

  student_id         text_source  cardiff_label  cardiff_offensive
0        1_A  caption+transcript  not_offensive           0.244071
1        1_A  caption+transcript  not_offensive           0.093976
2        1_A        caption_only  not_offensive           0.126799
3        2_A        caption_only  not_offensive           0.065943
4        2_A        caption_only  not_offensive           0.139014


In [7]:
print("=== Cardiff offensive label counts ===")
print(df["cardiff_label"].value_counts(dropna=False))
print()
print("=== Offensive posts (score > 0.5) ===")
offensive_only = df[df["cardiff_is_toxic"] == True]
print(f"Total offensive: {len(offensive_only)} / {df['has_text'].sum()} usable posts")
print()
print("=== Mean offensive score ===")
print(f"  All usable: {df.loc[df['has_text'], 'cardiff_offensive'].mean():.4f}")
if len(offensive_only) > 0:
    print(f"  Offensive posts only: {offensive_only['cardiff_offensive'].mean():.4f}")

=== Cardiff offensive label counts ===
cardiff_label
not_offensive    772
None             108
offensive         25
Name: count, dtype: int64

=== Offensive posts (score > 0.5) ===
Total offensive: 25 / 797 usable posts

=== Mean offensive score ===
  All usable: 0.1483
  Offensive posts only: 0.6689


In [8]:
print("=== Offensive posts by text source ===")
print(pd.crosstab(df["text_source"], df["cardiff_is_toxic"]))
print()
print("=== Offensive score by text source ===")
print(df.groupby("text_source")["cardiff_offensive"].agg(["mean", "median", "max", "count"]))

=== Offensive posts by text source ===
cardiff_is_toxic    False  True 
text_source                     
caption+transcript    127     16
caption_only          644      8
transcript_only         1      1

=== Offensive score by text source ===
                        mean    median       max  count
text_source                                            
caption+transcript  0.226866  0.171403  0.864322    143
caption_only        0.130085  0.102771  0.829398    652
none                     NaN       NaN       NaN      0
transcript_only     0.475506  0.475506  0.705816      2


In [9]:
offensive_only = df[df["cardiff_is_toxic"] == True].sort_values("cardiff_offensive", ascending=False)
print(f"=== Top {min(15, len(offensive_only))} offensive posts ===")
for _, r in offensive_only.head(15).iterrows():
    text_preview = r['text_for_analysis'][:200].replace('\n', ' | ')
    print(f"\n  [{r['cardiff_offensive']:.2f}] [{r['student_id']}] [{r['text_source']}]")
    print(f"  {text_preview}")

=== Top 15 offensive posts ===

  [0.86] [6_A] [caption+transcript]
  dirty blonde core #dirtyblonde #blondehair #naturalhair |  | People die for this, people lie for this, people suck and fuck some guy for this, | pay the toll for this, sell their soul for this, play my part

  [0.86] [12_A] [caption+transcript]
  GYMSKIN doubled his AURA after they didnt BURN THE BEAN ???? #gymskin |  | These are... Too groovy. Look at these. They didn't fucking burn the bean. Look at these fucking...

  [0.83] [6_A] [caption+transcript]
  ok but for real can we all agree that abby would be ilya and cathy would be shane? #abbyleemiller #heatedrivalry #edit #dancemomsedits |  | You suck my d***. Oh, that sounded really bratty. We can figure 

  [0.83] [10_A] [caption_only]
  "pulling up to talk shit with my mom and realizing she's not in the mood to take my side" "Pivot turn exit"

  [0.82] [1_A] [caption+transcript]
  The Supreme Court ruled trumps tariffs are illegal |  | Ready and turn. Nice. The 

In [10]:
# Load both previous toxicity outputs
detox_df = pd.read_csv("../Outputs/toxicity_detoxify_905.csv")
snlp_df = pd.read_csv("../Outputs/toxicity_sNLP_905.csv")

# Get boolean flags from all three models
detox_flag = detox_df["is_toxic"].fillna(False).astype(bool)
snlp_flag = snlp_df["snlp_is_toxic"].fillna(False).astype(bool)
cardiff_flag = df["cardiff_is_toxic"].fillna(False).astype(bool)

# How many posts each model flagged
print(f"Detoxify flagged: {detox_flag.sum()}")
print(f"s-nlp flagged: {snlp_flag.sum()}")
print(f"Cardiff flagged: {cardiff_flag.sum()}")
print()

# Three-way agreement
all_three = (detox_flag & snlp_flag & cardiff_flag).sum()
at_least_two = ((detox_flag.astype(int) + snlp_flag.astype(int) + cardiff_flag.astype(int)) >= 2).sum()
any_model = (detox_flag | snlp_flag | cardiff_flag).sum()

print(f"=== Three-model agreement ===")
print(f"All 3 models agree (highest confidence): {all_three}")
print(f"At least 2 models agree: {at_least_two}")
print(f"At least 1 model flags: {any_model}")
print()

# Pairwise overlap
print(f"=== Pairwise agreement ===")
print(f"Detoxify ∩ s-nlp: {(detox_flag & snlp_flag).sum()}")
print(f"Detoxify ∩ Cardiff: {(detox_flag & cardiff_flag).sum()}")
print(f"s-nlp ∩ Cardiff: {(snlp_flag & cardiff_flag).sum()}")

Detoxify flagged: 21
s-nlp flagged: 16
Cardiff flagged: 25

=== Three-model agreement ===
All 3 models agree (highest confidence): 12
At least 2 models agree: 17
At least 1 model flags: 33

=== Pairwise agreement ===
Detoxify ∩ s-nlp: 12
Detoxify ∩ Cardiff: 15
s-nlp ∩ Cardiff: 14


In [11]:
output_path = "../Outputs/toxicity_cardiff_offensive_905.csv"
df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: ../Outputs/toxicity_cardiff_offensive_905.csv


# Toxicity Model 3: Cardiff Twitter Offensive — Conclusion

**Model:** `cardiffnlp/twitter-roberta-base-offensive`
**Output labels:** Binary — offensive vs non-offensive
**Trained on:** 58 million tweets (domain match for social media)
**Threshold applied:** 0.5
**Dataset:** 905 posts from 12 students, 797 had usable text

## What we did in plain terms

Ran the third toxicity model. This one is trained specifically on tweets, so it should understand social media language better. Same as the other two, it gives a single score 0 to 1 for whether the post is offensive.

## What we found

Out of 797 usable posts, 25 (3.1%) crossed the 0.5 threshold and were labeled offensive. Cardiff is the most aggressive of the three toxicity models we ran:
- Detoxify: 21 posts flagged
- s-nlp: 16 posts flagged
- Cardiff Twitter: 25 posts flagged

The transcript effect is consistent with the other models:
- Caption only: 8 toxic out of 652 (1.2%)
- Caption + transcript: 16 toxic out of 143 (11.2%)
- Transcript only: 1 toxic out of 2

About 10x higher flag rate with transcripts. All three models confirm this finding.

## Cross-model agreement (the key result for the toxicity section)

Comparing all three models on the same 797 posts:

| Confidence level | Posts |
|---|---|
| All 3 models agree | 12 |
| At least 2 models agree | 17 |
| At least 1 model flags | 33 |
| Q8 students flagged as uncivil | 10 |

The "all 3 agree" count of 12 is nearly identical to the 10 posts students themselves flagged as uncivil in Q8. When the three models converge, the count matches human perception.

Pairwise overlap is high too:
- Detoxify ∩ s-nlp: 12 shared
- Detoxify ∩ Cardiff: 15 shared
- s-nlp ∩ Cardiff: 14 shared

This tells us the three models are detecting roughly the same content, just with slightly different sensitivity. Cardiff is the most aggressive, s-nlp the most conservative, Detoxify in the middle.

## What Cardiff catches that others don't

Cardiff is more sensitive to politically charged or aggressive vocabulary even without profanity. For example, an airport security PSA ("I don't care if you're white, I don't care if you're a citizen") got flagged 71% offensive by Cardiff even though the message is benign. Cardiff is reading the racial framing and aggressive tone as offensive, while the other models see no profanity and rate it low.

This is both a strength (catches subtle aggression) and a weakness (false positives on PSAs and political content).

## The remaining problem (still profanity ≠ toxicity)

Even with three models agreeing, the 12 "consensus toxic" posts are almost all entertainment content with casual profanity:
- A gym influencer saying "didn't fucking burn the bean"
- Music lyrics with explicit content
- A pet vlog with someone saying "what the f***"
- TV show clips with mild swearing

None of these are genuinely harmful. The shared definition all three models have learned is "text containing toxic-flagged vocabulary," not "text that actually harms or harasses."

## Bottom line and methodology recommendation

**For per-post toxicity labels, use multi-model agreement as a confidence filter:**
- "Toxic" = at least 2 of 3 models flag → 17 posts
- "Highly confident toxic" = all 3 models flag → 12 posts
- "Clean" = no models flag → 764 posts

The "all 3 agree" set of 12 closely matches the 10 posts students flagged in Q8 (which measures perceived incivility). This is the strongest validation we have — model consensus aligns with human perception.

But all three models share a common limitation: they detect profanity, not harm. To distinguish "casual entertainment swearing" from "actual harassment," LLM-based classification is the recommended next step.